# Qwen3-ASR A2S Fast Fine-tune

唯一执行路径：公开 gold transcript、单 BF16 LoRA adapter、先 smoke 再完整训练。

In [ ]:
from google.colab import drive
from pathlib import Path
import os
import subprocess

drive.mount('/content/drive')
PROJECT = Path('/content/mega-asr')
if not (PROJECT / '.git').is_dir():
    subprocess.run(['git', 'clone', 'https://github.com/cluster1900/lora-asr.git', str(PROJECT)], check=True)
else:
    subprocess.run(['git', '-C', str(PROJECT), 'fetch', 'origin', 'main'], check=True)
    subprocess.run(['git', '-C', str(PROJECT), 'checkout', 'main'], check=True)
    subprocess.run(['git', '-C', str(PROJECT), 'pull', '--ff-only', 'origin', 'main'], check=True)
os.chdir(PROJECT)
print('repo_commit=', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())

In [ ]:
%pip install -q -r requirements-colab.txt
%pip install -q flash-attn==2.8.3.post1 --no-build-isolation

In [ ]:
DRIVE_ROOT = Path('/content/drive/MyDrive/qwen3-asr-public-a2s')
LOCAL_DATA = Path('/content/qwen3-asr-runtime/data')
SMOKE_CANDIDATES = DRIVE_ROOT / 'candidates/smoke'
FULL_CANDIDATES = DRIVE_ROOT / 'candidates/full'
MANIFESTS = DRIVE_ROOT / 'manifests'
BASE = DRIVE_ROOT / 'base'
RESULTS = DRIVE_ROOT / 'results'
SMOKE_RUN = DRIVE_ROOT / 'runs/smoke-v1'
for path in (LOCAL_DATA, SMOKE_CANDIDATES, FULL_CANDIDATES, MANIFESTS, BASE, RESULTS):
    path.mkdir(parents=True, exist_ok=True)

DATA_CONFIG = PROJECT / 'configs/data/public_robust_200k.yaml'
TRAIN_CONFIG = PROJECT / 'configs/train/qwen3_asr_public_200k_a2s.yaml'
DATA_TOOL = PROJECT / 'scripts/prepare_public_robust_manifests.py'
INFER = PROJECT / 'inference/qwen3_asr_infer.py'
EVAL = PROJECT / 'evaluation/eval_wer.py'
TRAIN = PROJECT / 'train/train_qwen3_asr_a2s.py'

def run(*parts, check=True):
    command = [str(part) for part in parts]
    print('$', ' '.join(command))
    return subprocess.run(command, cwd=PROJECT, check=check, text=True)

## 1. Metadata 与 128-row smoke

In [ ]:
run('python', DATA_TOOL, 'probe', '--config', DATA_CONFIG, '--output', DRIVE_ROOT / 'probe.json')
run('python', DATA_TOOL, 'stage', '--config', DATA_CONFIG, '--mode', 'smoke',
    '--candidate-dir', SMOKE_CANDIDATES, '--data-root', LOCAL_DATA)
run('python', DATA_TOOL, 'smoke', '--config', DATA_CONFIG,
    '--robust-candidates', SMOKE_CANDIDATES / 'robust.jsonl',
    '--english-clean-candidates', SMOKE_CANDIDATES / 'english_clean.jsonl',
    '--chinese-clean-candidates', SMOKE_CANDIDATES / 'chinese_clean.jsonl',
    '--bench-candidates', SMOKE_CANDIDATES / 'bench.jsonl',
    '--output-dir', MANIFESTS, '--data-root', LOCAL_DATA, '--audio-mode', 'decode', '--force')

In [ ]:
SMOKE_MANIFEST = MANIFESTS / 'public_robust_smoke_128.jsonl'
SMOKE_PREDICTIONS = BASE / 'smoke_128.predictions.jsonl'
run('python', INFER, '--manifest', SMOKE_MANIFEST, '--output-jsonl', SMOKE_PREDICTIONS,
    '--audio-root', LOCAL_DATA, '--resume')
run('python', EVAL, '--predictions-jsonl', SMOKE_PREDICTIONS, '--output-dir', BASE / 'smoke_128')

import json
rows = [json.loads(line) for line in SMOKE_PREDICTIONS.read_text(encoding='utf-8').splitlines() if line]
successful = [row for row in rows if not row.get('error')]
assert any(row.get('condition_group') == 'clean' for row in successful)
assert any(row.get('condition_group') in {'atomic', 'compound'} for row in successful)

## 2. 10+2 step checkpoint/resume 门禁

In [ ]:
run('python', TRAIN, '--config', TRAIN_CONFIG, '--validate-only', '--print-plan')
run('python', TRAIN, '--config', TRAIN_CONFIG, '--output-dir', SMOKE_RUN, '--smoke-steps', '10')
run('python', TRAIN, '--config', TRAIN_CONFIG, '--output-dir', SMOKE_RUN,
    '--smoke-steps', '12', '--resume', 'auto')

## 3. 按配额 staging 完整数据

In [ ]:
run('python', DATA_TOOL, 'stage', '--config', DATA_CONFIG, '--mode', 'full',
    '--candidate-dir', FULL_CANDIDATES, '--data-root', LOCAL_DATA)
run('python', DATA_TOOL, 'build', '--config', DATA_CONFIG,
    '--robust-candidates', FULL_CANDIDATES / 'robust.jsonl',
    '--english-clean-candidates', FULL_CANDIDATES / 'english_clean.jsonl',
    '--chinese-clean-candidates', FULL_CANDIDATES / 'chinese_clean.jsonl',
    '--bench-candidates', FULL_CANDIDATES / 'bench.jsonl',
    '--output-dir', MANIFESTS, '--data-root', LOCAL_DATA, '--audio-mode', 'decode', '--force')

## 4. 最少量 BF16 base 评分并生成 30k curriculum

In [ ]:
TRAIN_MANIFEST = MANIFESTS / 'public_robust_200k_train.jsonl'
BASE_TRAIN_PREDICTIONS = BASE / 'train_curriculum.predictions.jsonl'
CURRICULUM = MANIFESTS / 'public_robust_30k_curriculum.jsonl'
for limit in (60000, 100000, 160000, 200000):
    run('python', INFER, '--manifest', TRAIN_MANIFEST, '--output-jsonl', BASE_TRAIN_PREDICTIONS,
        '--audio-root', LOCAL_DATA, '--limit', limit, '--resume')
    run('python', EVAL, '--predictions-jsonl', BASE_TRAIN_PREDICTIONS,
        '--output-dir', BASE / 'train_curriculum')
    built = run('python', DATA_TOOL, 'curriculum', '--config', DATA_CONFIG,
        '--train', TRAIN_MANIFEST, '--scored', BASE / 'train_curriculum/scored.jsonl',
        '--output', CURRICULUM, '--report', MANIFESTS / 'public_robust_30k_curriculum.report.json',
        '--force', check=False)
    if built.returncode == 0:
        print('curriculum_ready_at_limit=', limit)
        break
else:
    raise RuntimeError('200k base predictions still cannot produce the fixed 30k curriculum')

## 5. 固定 base canary、validation 与 Bench baseline

In [ ]:
BASE_JOBS = {
    'validation_canary_512': MANIFESTS / 'public_robust_512_canary.jsonl',
    'validation_10k': MANIFESTS / 'public_robust_10k_val.jsonl',
    'bench_5k': MANIFESTS / 'vitw_bench_5k_test.jsonl',
}
for name, manifest in BASE_JOBS.items():
    predictions = BASE / f'{name}.predictions.jsonl'
    run('python', INFER, '--manifest', manifest, '--output-jsonl', predictions,
        '--audio-root', LOCAL_DATA, '--resume')
    run('python', EVAL, '--predictions-jsonl', predictions, '--output-dir', BASE / name)

## 6. 单 adapter 三阶段训练与 release 评测

In [ ]:
run('python', TRAIN, '--config', TRAIN_CONFIG, '--resume', 'auto')
ADAPTER = DRIVE_ROOT / 'runs/main/release/adapter'
assert ADAPTER.is_dir(), ADAPTER
for name in ('validation_10k', 'bench_5k'):
    manifest = BASE_JOBS[name]
    predictions = RESULTS / f'{name}.predictions.jsonl'
    run('python', INFER, '--manifest', manifest, '--output-jsonl', predictions,
        '--adapter-dir', ADAPTER, '--audio-root', LOCAL_DATA, '--resume')
    run('python', EVAL, '--predictions-jsonl', predictions, '--output-dir', RESULTS / name)